In [3]:
import sys
import os

# Adds 'src/capstone' to the path so 'pipeline' is discoverable
# assuming the notebook is in ROOT/notebooks/ and code is in ROOT/src/capstone/
sys.path.append(os.path.abspath("../"))

import pandas as pd

from pipeline.version_config import VersionConfig
from pipeline.pipeline_run import PipelineRun
from pipeline.factory import PipelineFactory

## Config

`use_synthetic=True, target_real_pct=5/6` → SyntheticAugmenter appends ~20% synthetic
rows to `X_train` (formula: `floor(real / target_real_pct) - real`).
No snapshot flags set → `build()` leaves all version counters unchanged.
This run will read from GCS but write nothing back.

In [4]:
config = (
    VersionConfig.load(use_synthetic=False)
    .snapshot_raw()
    .snapshot_final()
    .snapshot_models_new_data()    # model major bump v3.2 → v4.0
    .snapshot_final()
    .build()
)
run = PipelineRun(config)
stages = PipelineFactory.full_run(config)

print('\nScenario:', stages.scenario)
print('current data raw_version   :', config.raw_version)
print('current model_version :', config.model_version)

No versions.json found in GCS — using defaults. Call config.commit() after your first snapshot to persist versions.
VersionConfig loaded:
  data:          v3.2 (raw_suffix='real')
  baselines:     v3.2
  model:         v4.0
  hyperparams:   v1.0
  use_synthetic: False

VersionConfig ready:
  Active flags      : ['raw', 'final', 'models', 'model_major', 'baselines']
  data              : v3.2 -> v3.3
  baselines         : v3.2 -> v4.0
  model             : v4.0 -> v5.0
  hyperparams       : v1.0 (unchanged)
  raw_version       : v3.2_real  ->  v3.3_real
  next_final_version: v3.3_100real
  model_version     : v4.0  ->  v5.0
  baselines_version : v3.2  ->  v4.0
  hyperparam_version: v1.0
  use_synthetic     : False

  Call config.commit() after all snapshots succeed.

Scenario: full_run
current data raw_version   : v3.2_real
current model_version : v4.0


---
## Main Flow

Runs the full `retrain_existing_data` scenario with synthetic augmentation.

## Stage 1 — DataLoader (GCS)

Reads the parquet snapshot at `config.raw_version`. No BigQuery call.

In [5]:
stages.loader.run(run)
run.summary()

Pulling video_snapshots from BQ (maduros-dolce.capstone_youtube)...


/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  60696 video rows
Pulling channel_baseline_videos from BQ...
  28814 baseline rows (974 channels)
Pulling channel_baseline_medians from BQ...
  974 baseline median rows
PipelineRun(config=VersionConfig(data=(3, 3), model=(5, 0), baselines=(4, 0), hyperparams=(1, 0), use_synthetic=False, active=['raw', 'final', 'models', 'model_major', 'baselines']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       None
  df_engineered  None
  df_train       None
  df_test        None
  df_val         None
  X_train        None
  X_test         None
  X_val          None
  X_val_unscaled  None
  y_train        None
  y_test         None
  y_val          None
  models         empty dict
  results        empty dict


## Stage 2 — DataPreprocessor

Pivots the long-format poll records into one row per video (via `pivot_snapshots`),
joins channel baselines, and runs structural cleanup via `build_clean_dataset`.
Writes `run.df_clean`.

In [6]:
stages.preprocessor.run(run)
run.summary()

Building clean dataset
snapshot cols: Index(['video_id', 'poll_timestamp', 'channel_id', 'channel_handle', 'title',
       'view_count', 'like_count', 'comment_count', 'face_count', 'brightness',
       'colorfulness', 'vertical', 'tier', 'description', 'tags',
       'duration_seconds', 'category_id', 'category_name', 'published_at',
       'poll_label', 'hours_since_publish', 'subscriber_count',
       'contains_synthetic_media'],
      dtype='object')

[1/3] Pivoting snapshots...
  Videos with all 3 polls: 18334 (dropped 3111 incomplete)
  Pivoted shape: (18403, 34)

[2/3] Joining baseline medians...
  Baseline join: 18403/18403 videos matched a channel median

[3/3] Cleaning data...
  Cleaned: 18403 rows × 40 columns

Clean dataset: 18403 rows × 40 columns
PipelineRun(config=VersionConfig(data=(3, 3), model=(5, 0), baselines=(4, 0), hyperparams=(1, 0), use_synthetic=False, active=['raw', 'final', 'models', 'model_major', 'baselines']))
  df_videos      populated  DataFrame shape=(6

## Stage 3 — FeatureEngineer

Runs the full feature engineering chain on `run.df_clean` (one row per video).
Computes `above_baseline` target, all ratio/growth features, and categorical encodings.
Writes `run.df_engineered`.

In [13]:
# TODO(jelani): df_clean doesn't filter for unique video IDs (and should). Look
# into why there are duplicate rows.
print(run.df_clean['video_id'].nunique())

18334


In [12]:
stages.engineer.run(run)
run.summary()

  all: dropped 414 rows with NaN in a baseline_median_* column or 0.0 baseline_median_engagement_rate
Engineering features

[1/9] Computing target variable...
  Target distribution: 55.1% above baseline, 44.9% below

[2/9] Computing velocity features...
  Computed velocity, upload momentum, normalized velocity, and acceleration features

[3/9] Computing ratio and baseline-normalized features...
  Computed ratio and baseline-normalized features

[4/9] Computing subscriber-normalized metrics...
  Computed subscriber-normalized metrics for upload/24h/7d

[5/9] Computing categorical features...
  Title categories:
title_category
neutral        11344
all_caps        2048
exclamation     1955
question        1572
listicle         562
how_to           230
clickbait        204
emoji_heavy       74
  Description categories:
desc_category
link_heavy        4729
minimal           4509
has_links         3837
has_hashtags      2697
neutral           1395
has_timestamps     774
long_form           4

## Stage 4 — DataSplitter

Loads the locked validation IDs from GCS (`splits/validation_ids.json`).
Filters `run.df_engineered` to produce `df_val`, then stratifies the remaining
rows into `df_train` / `df_test` (80/20). Derives unscaled `X_train`, `X_test`,
`X_val` and corresponding `y_` series.

In [14]:
stages.splitter.run(run)
run.summary()

DataSplitter — loaded holdout (5,193 val rows):
  df_val:    5,193 rows (28.9%)
  df_train: 10,236 rows (56.9%)
  df_test:   2,560 rows (14.2%)
PipelineRun(config=VersionConfig(data=(3, 3), model=(5, 0), baselines=(4, 0), hyperparams=(1, 0), use_synthetic=False, active=['raw', 'final', 'models', 'model_major', 'baselines']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 87)
  df_train       populated  DataFrame shape=(10236, 87)
  df_test        populated  DataFrame shape=(2560, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(10236, 53)
  X_test         populated  DataFrame shape=(2560, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  None
  y_train        populated  Series length

## Stage 5 — Scaler

Fits `StandardScaler` on `X_train`, transforms `X_train`, `X_test`, and `X_val`.
Captures `run.X_val_unscaled` so Validator can apply each loaded model's own
historical scaler for a fair apples-to-apples comparison.

In [15]:
stages.scaler.run(run)
run.summary()

PipelineRun(config=VersionConfig(data=(3, 3), model=(5, 0), baselines=(4, 0), hyperparams=(1, 0), use_synthetic=False, active=['raw', 'final', 'models', 'model_major', 'baselines']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 87)
  df_train       populated  DataFrame shape=(10236, 87)
  df_test        populated  DataFrame shape=(2560, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(10236, 53)
  X_test         populated  DataFrame shape=(2560, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  populated  DataFrame shape=(5193, 53)
  y_train        populated  Series length=10236
  y_test         populated  Series length=2560
  y_val          populated  Series length=5193
  models  

## Stage 6 — SyntheticAugmenter

Generates synthetic rows from `run.df_train` via `generate_synthetic_data`,
engineers them through the same `FeatureEngineerLogic` chain, aligns with
`X_train`'s column space, scales with the fitted `StandardScaler`, then
concatenates onto `run.X_train` / `run.y_train`. `X_test` and `X_val` are
never touched. Target: ~20% of real train rows added as synthetic
(`target_real_pct=5/6` → `floor(real/0.8333) - real ≈ 0.2 * real`).

In [12]:
stages.augmenter.run(run)
run.summary()

[SyntheticAugmenter] use_synthetic=False — skipping.
PipelineRun(config=VersionConfig(data=(3, 1), model=(4, 0), hyperparams=(1, 0), use_synthetic=False, active=['models', 'model_major']))
  df_videos      populated  DataFrame shape=(58037, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(17532, 40)
  df_engineered  populated  DataFrame shape=(17136, 87)
  df_train       populated  DataFrame shape=(9554, 87)
  df_test        populated  DataFrame shape=(2389, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(9554, 53)
  X_test         populated  DataFrame shape=(2389, 53)
  X_val          populated  DataFrame shape=(5193, 53)
  X_val_unscaled  populated  DataFrame shape=(5193, 53)
  y_train        populated  Series length=9554
  y_test         populated  Series length=2389
  y_val          populated  Series length=5193
  model

## Stage 7 — Trainer
train all models + evaluate on x_test. Populates run.models

In [16]:
stages.trainer.run(run)
run.summary()

Loaded hyperparams 'v1.0' (saved 2026-04-21)
  Models: ['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier']
  Search: {'strategy': 'none', 'note': 'pre-tuning baseline'}
Loaded hyperparams from snapshot 'v1.0'.
Training LogisticRegression (L1)...
Training RandomForestClassifier...
Training XGBClassifier...
Training VotingClassifier ensemble (RF + XGB, weights=[1, 2])...

=== ModelTrainer — test-set results ===
  lr_l1         AUC=0.7537  acc=0.6895  F1↑=0.7231
  rf            AUC=0.8552  acc=0.7816  F1↑=0.8108
  xgb           AUC=0.9060  acc=0.8230  F1↑=0.8409
  ensemble      AUC=0.9018  acc=0.8234  F1↑=0.8427
PipelineRun(config=VersionConfig(data=(3, 3), model=(5, 0), baselines=(4, 0), hyperparams=(1, 0), use_synthetic=False, active=['raw', 'final', 'models', 'model_major', 'baselines']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       popu

## Stage 8 — ModelSnapshotter

writes the v4.0_* models

In [17]:
stages.model_snapshotter.run(run)
print('\nLoaded models:', list(run.models.keys()))

Model artifacts saved to models/v5.0_lr_l1/
  Uploaded gs://maduros-dolce-capstone-data/models/v5.0_lr_l1/model.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.0_lr_l1/scaler.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.0_lr_l1/feature_cols.json
  Uploaded gs://maduros-dolce-capstone-data/models/v5.0_lr_l1/metadata.json

Model v5.0_lr_l1 (LogisticRegression)
  Data: v3.3_100real (10236 real + 0 synthetic)
  ROC-AUC: 0.7537
  Accuracy: 0.6895
  F1 (above): 0.7231
Model artifacts saved to models/v5.0_rf/
  Uploaded gs://maduros-dolce-capstone-data/models/v5.0_rf/model.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.0_rf/scaler.pkl
  Uploaded gs://maduros-dolce-capstone-data/models/v5.0_rf/feature_cols.json
  Uploaded gs://maduros-dolce-capstone-data/models/v5.0_rf/metadata.json

Model v5.0_rf (RandomForest)
  Data: v3.3_100real (10236 real + 0 synthetic)
  ROC-AUC: 0.8552
  Accuracy: 0.7816
  F1 (above): 0.8108
Model artifacts saved to models/v5.0_xgb/
  

## Stage 8 — Validator

Evaluates each loaded model on the locked validation set (`X_val_unscaled` +
each model's own historical scaler). Populates `run.results` with AUC, accuracy,
F1, precision, recall, and top feature importances per model.

In [18]:
stages.validator.run(run)
run.summary()


=== Validator — validation-set results ===
  lr_l1           AUC=0.7552  acc=0.6913  F1↑=0.7259
  rf              AUC=0.8642  acc=0.7901  F1↑=0.8175
  xgb             AUC=0.9055  acc=0.8251  F1↑=0.8438
  ensemble        AUC=0.9031  acc=0.8269  F1↑=0.8459
PipelineRun(config=VersionConfig(data=(3, 3), model=(5, 0), baselines=(4, 0), hyperparams=(1, 0), use_synthetic=False, active=['raw', 'final', 'models', 'model_major', 'baselines']))
  df_videos      populated  DataFrame shape=(60696, 23)
  df_baselines   populated  DataFrame shape=(28814, 15)
  df_medians     populated  DataFrame shape=(974, 7)
  df_clean       populated  DataFrame shape=(18403, 40)
  df_engineered  populated  DataFrame shape=(17989, 87)
  df_train       populated  DataFrame shape=(10236, 87)
  df_test        populated  DataFrame shape=(2560, 87)
  df_val         populated  DataFrame shape=(5193, 87)
  X_train        populated  DataFrame shape=(10236, 53)
  X_test         populated  DataFrame shape=(2560, 53)
  X_val

## Stage 9 — Persist Validation Results to GCS

In [19]:
stages.validation_results_snapshotter.run(run)
# Appends to gs://maduros-dolce-capstone-data/models/{model_version}/validation_results.jsonl

Validation results appended → gs://maduros-dolce-capstone-data/models/v5.0/validation_results.jsonl
  lr_l1           AUC=0.7552  acc=0.6913  F1↑=0.7259
  rf              AUC=0.8642  acc=0.7901  F1↑=0.8175
  xgb             AUC=0.9055  acc=0.8251  F1↑=0.8438
  ensemble        AUC=0.9031  acc=0.8269  F1↑=0.8459


PipelineRun(model_version='v4.0', num_synth_rows=0, populated=['df_videos', 'df_baselines', 'df_medians', 'df_clean', 'df_engineered', 'df_train', 'df_test', 'df_val', 'X_train', 'X_test', 'X_val', 'X_val_unscaled', 'y_train', 'y_test', 'y_val', 'models', 'results'])

## Results

This is the **first validation-set evaluation** for this project — there is no prior
validation baseline in `data_analysis.ipynb` (only test-set scores were recorded there).

Cross-check model rank order and approximate AUC range against the test scores
from `data_analysis.ipynb`. Validation AUC will typically be slightly lower than
test AUC since the holdout was never touched during training.

In [20]:
metric_cols = [
    'roc_auc', 'accuracy',
    'f1_above', 'precision_above', 'recall_above',
    'f1_below', 'precision_below', 'recall_below',
]

df_results = (
    pd.DataFrame(run.results).T
    [metric_cols]
    .astype(float)
    .round(4)
)
df_results.index.name = 'model'

df_results.style.highlight_max(color='#d4edda').format('{:.4f}')

,roc_auc,accuracy,f1_above,precision_above,recall_above,f1_below,precision_below,recall_below
model,,,,,,,,
lr_l1,0.7552,0.6913,0.7259,0.7160,0.7361,0.6467,0.6584,0.6353
rf,0.8642,0.7901,0.8175,0.7905,0.8464,0.7531,0.7895,0.7198
xgb,0.9055,0.8251,0.8438,0.8372,0.8506,0.8014,0.8095,0.7934
ensemble,0.9031,0.8269,0.8459,0.8366,0.8554,0.8025,0.8142,0.7913


In [21]:
# Top features for each model — sanity check against data_analysis.ipynb
for name, entry in run.results.items():
    top = entry.get('top_features', [])[:5]
    print(f'\n{name}:')
    for f in top:
        key = 'coefficient' if 'coefficient' in f else 'importance'
        print(f'  {f["feature"]:35s}  {f[key]:+.4f}')


lr_l1:
  like_count_upload_vs_baseline        +9.8967
  view_count_upload_vs_baseline        -2.1114
  like_rate_24h                        +1.2251
  view_count_24h                       -0.3407
  like_rate_upload                     -0.2937

rf:
  like_rate_24h                        +0.1317
  view_velocity_ratio                  +0.0528
  like_rate_upload                     +0.0441
  like_count_upload_vs_baseline        +0.0438
  duration_seconds                     +0.0341

xgb:
  baseline_baseline_video_count        +0.0665
  is_short                             +0.0638
  like_rate_24h                        +0.0548
  tier_encoded                         +0.0547
  view_count_upload_vs_baseline        +0.0246

ensemble:


# 3-Way (Train, Test, Validation) Evaluation

`hist_test` = test metrics from original v3.1 training.
`curr_test` = same loaded models on today's test split.
`val` = locked holdout. 

TODO: To store to GCS long-term, extend ValidationResultsSnapshotter to also snapshot test_results_current

In [23]:
# Current test-set evaluation of loaded models (requires reconstructing X_test_unscaled)
import pandas as pd
from utils.snapshot_model import ModelResult
from dataclasses import asdict

X_test_unscaled = pd.DataFrame(
    stages.scaler.scaler_.inverse_transform(run.X_test),
    columns=run.X_test.columns,
    index=run.X_test.index,
)

test_results_current = {}
for name, entry in run.models.items():
    X = stages.validator.logic._prepare_X(entry, X_test_unscaled)
    result = ModelResult.from_sklearn(entry["model"], X, run.y_test, X.columns.tolist())
    test_results_current[name] = asdict(result)

metric_cols = ['roc_auc', 'accuracy', 'f1_above', 'precision_above', 'recall_above',
               'f1_below', 'precision_below', 'recall_below']

records = {}
for name in run.results:
    print(name)
    print(run.results)
    if ("metadata" in run.models[name]):
        records[(name, 'hist_test')] = {m: run.models[name]["metadata"]["result"].get(m) for m in metric_cols}
    records[(name, 'curr_test')] = {m: test_results_current[name].get(m) for m in metric_cols}
    records[(name, 'val')]       = {m: run.results[name].get(m) for m in metric_cols}

df_3way = (
    pd.DataFrame(records).T
    .astype(float)
    .round(4)
)
df_3way.index.names = ['model', 'split']
df_3way.style.highlight_max(color='#d4edda').format('{:.4f}')

lr_l1
{'lr_l1': {'roc_auc': np.float64(0.7552), 'accuracy': 0.6913, 'precision_above': 0.716, 'recall_above': 0.7361, 'f1_above': 0.7259, 'precision_below': 0.6584, 'recall_below': 0.6353, 'f1_below': 0.6467, 'confusion_matrix': [[1467, 842], [761, 2123]], 'top_features': [{'feature': 'like_count_upload_vs_baseline', 'coefficient': 9.896712}, {'feature': 'view_count_upload_vs_baseline', 'coefficient': -2.111368}, {'feature': 'like_rate_24h', 'coefficient': 1.22508}, {'feature': 'view_count_24h', 'coefficient': -0.340686}, {'feature': 'like_rate_upload', 'coefficient': -0.293682}, {'feature': 'view_count_velocity_24h', 'coefficient': 0.235084}, {'feature': 'duration_bucket', 'coefficient': 0.216192}, {'feature': 'is_long', 'coefficient': -0.211258}, {'feature': 'like_count_velocity_24h', 'coefficient': -0.211099}, {'feature': 'like_count_24h', 'coefficient': 0.205606}, {'feature': 'title_category', 'coefficient': -0.183}, {'feature': 'view_velocity_ratio', 'coefficient': -0.163246}, {'f

# Final: GCS Write 

Commit config changes to GCS.

In [24]:
config.commit()

Committed versions.json -> data v3.3, model v5.0, baselines v4.0, hyperparams v1.0


---
# Hyperparameter Tuning — Grid Search on v4.0 Models

Runs `RandomizedSearchCV` over LR, RF, and XGB using the same data splits
as the v4.0 training run (no GCS reload). Saves tuned params as a new
hyperparam version and snapshots model artifacts + validation results.

In [25]:
# Set False only when you're ready to write to GCS.
dry_run = True

## Tuning Config

In [26]:
config_tune = (
    VersionConfig.load(use_synthetic=False)
    .tune(strategy="random", n_iter=50, cv=5)
    .snapshot_hyperparams()
    .snapshot_models()
    .build()
)
run_tune = PipelineRun(config_tune)
stages_tune = PipelineFactory.tune_hyperparams(config_tune)

print('\nScenario         :', stages_tune.scenario)
print('model_version    :', config_tune.model_version)
print('hyperparam_version:', config_tune.hyperparam_version)

VersionConfig loaded:
  data:          v3.3 (raw_suffix='real')
  baselines:     v4.0
  model:         v5.0
  hyperparams:   v1.0
  use_synthetic: False

VersionConfig ready:
  Active flags      : ['models', 'hyperparams', 'tune']
  data              : v3.3 (unchanged)
  baselines         : v4.0 (unchanged)
  model             : v5.0 -> v5.1
  hyperparams       : v1.0 -> v1.1
  raw_version       : v3.3_real
  next_final_version: v3.3_100real
  model_version     : v5.0  ->  v5.1
  baselines_version : v4.0
  hyperparam_version: v1.0  ->  v1.1
  use_synthetic     : False
  Tuning            : strategy=random, n_iter=50, cv=5, scoring=roc_auc

  Call config.commit() after all snapshots succeed.

Scenario         : tune_hyperparams
model_version    : v5.0
hyperparam_version: v1.0


## Reuse v4.0 Data Splits

Copy already-scaled splits from `run` so we skip stages 1–6 entirely.
Wire the fitted scaler so `ModelTrainer` can stamp it onto new model entries.

In [28]:
run_tune.X_train = run.X_train
run_tune.X_test  = run.X_test
run_tune.X_val   = run.X_val
run_tune.X_val_unscaled = run.X_val_unscaled
run_tune.y_train = run.y_train
run_tune.y_test  = run.y_test
run_tune.y_val   = run.y_val

# ModelTrainer stamps scaler_ onto each model entry; forward the v4.0 fit.
stages_tune.scaler.scaler_       = stages.scaler.scaler_
stages_tune.scaler.feature_cols_ = stages.scaler.feature_cols_

## Trainer (Grid Search)

With `config_tune.tune_models=True`, `ModelTrainer` runs `RandomizedSearchCV`
on LR, RF, and XGB before fitting the final models on the full train split. Search isn't needed for
Ensemble because it is using search params from RF and XGB.

In [ ]:
stages_tune.trainer.run(run_tune)
run_tune.summary()

Loaded hyperparams 'v1.0' (saved 2026-04-21)
  Models: ['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier']
  Search: {'strategy': 'none', 'note': 'pre-tuning baseline'}
Loaded hyperparams from snapshot 'v1.0'.
Tuning LogisticRegression with strategy='random' (cv=5, scoring='roc_auc')
  Sampling 50 combinations from grid of ~16 total
Fitting 5 folds for each of 16 candidates, totalling 80 fits


/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 16 is smaller than n_iter=50. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/jelanigould-bailey/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: 

## HyperparamSnapshotter — Persist Tuned Params

In [ ]:
if not dry_run:
    stages_tune.hyperparam_snapshotter.run(run_tune)
else:
    print('[dry_run] skipping HyperparamSnapshotter')

## Model Snapshotter — Save Tuned Models

In [ ]:
if not dry_run:
    stages_tune.model_snapshotter.run(run_tune)
else:
    print('[dry_run] skipping ModelSnapshotter')

## Validator — Evaluate on Locked Holdout

In [ ]:
stages_tune.validator.run(run_tune)
run_tune.summary()

## Validation Results Snapshotter

In [ ]:
if not dry_run:
    stages_tune.validation_results_snapshotter.run(run_tune)
    # Appends to gs://maduros-dolce-capstone-data/models/{model_version}/validation_results.jsonl
else:
    print('[dry_run] skipping ValidationResultsSnapshotter')

## Pre vs Post Tuning Comparison

Side-by-side validation metrics: default hyperparams (v4.0) vs tuned (v4.1).

In [ ]:
metric_cols = [
    'roc_auc', 'accuracy',
    'f1_above', 'precision_above', 'recall_above',
    'f1_below', 'precision_below', 'recall_below',
]

records = {}
for name in run.results:
    records[(name, 'v4.0_baseline')] = {m: run.results[name].get(m) for m in metric_cols}
for name in run_tune.results:
    records[(name, f'{config_tune.model_version}_tuned')] = {m: run_tune.results[name].get(m) for m in metric_cols}

df_compare = (
    pd.DataFrame(records).T
    .astype(float)
    .round(4)
)
df_compare.index.names = ['model', 'run']
df_compare.style.highlight_max(color='#d4edda').format('{:.4f}')

## Commit Tuning Config to GCS

In [ ]:
if not dry_run:
    config_tune.commit()
else:
    print('[dry_run] skipping config_tune.commit()')